# ACT benchmark 
Validates the first version of the taxonomy (37 action types) by comparing the assignments of
different classifiers with the ground truth, the expression dictionary that the author built by
hand during the construction of the taxonomy.

The classifiers are:
1. Majority class
2. Character n-gram + logistic regression
3. Japanese WordNet
4. Tohoku BERT v3 (kNN and nearest centroid), plus K-means clustering
5. LLMs, given the names of the types and representative expressions, without demonstrations or training data

The corpus (`event_actions.csv`), the ground-truth
dictionary (`ground_truth_v1.json`) and the response files are to be placed in `DATA_DIR` to run the notebook.

## Setup

In [ ]:
# Import necessary libraries
import json
import re
import unicodedata
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_mutual_info_score, adjusted_rand_score,
                             confusion_matrix, f1_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn

RANDOM_STATE = 42
N_SPLITS = 5
MIN_CLASS_SIZE = 5
K_NEIGHBORS = 5
pd.set_option("display.max_rows", 60)

In [ ]:
# Paths. 
DATA_DIR = Path("data")
CORPUS_CSV = DATA_DIR / "event_actions.csv"
GT_JSON = DATA_DIR / "ground_truth_v1.json"
PROMPT_FILE = Path("prompt.md")          # the prompt given to the LLMs 
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

SRC_SEKAISHI = "世界史図録"
SRC_FUNSO = "紛争事典"
for p in [CORPUS_CSV, GT_JSON, PROMPT_FILE]:
    print(f"{p}: exists={p.exists()}")

In [ ]:
# Normalise old character forms into unicode NFKC form
VARIANT_MAP = {
    "擊": "撃", "說": "説", "騷": "騒", "眞": "真", "國": "国", "會": "会",
    "戰": "戦", "廢": "廃", "舊": "旧", "來": "来", "發": "発", "獨": "独",
    "體": "体", "區": "区", "黨": "党", "團": "団",
}

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s).strip())
    return "".join(VARIANT_MAP.get(c, c) for c in s)

In [ ]:
# Load and preprocess the corpus

raw = pd.read_csv(CORPUS_CSV, encoding="utf-8-sig")
raw.columns = ["reference", "entity_name", "action"]
raw["action_norm"] = raw["action"].map(normalize)

print("rows (events):", len(raw))
print(raw.reference.value_counts().to_string())
for ref, g in raw.groupby("reference"):
    print(f"  {ref}: {g.action_norm.nunique()} unique normalized expressions")

## Load the ground truth

In [ ]:
# The ground truth maps each expression to one or more of the 37 action types.
# The author built it by hand during the construction of the taxonomy. It is not evaluated as a
# classifier: an exact-match lookup scores 0% on unseen expressions by construction.
VERB_TO_AT = json.loads(GT_JSON.read_text(encoding="utf-8"))

VERB_DICT = {}
for k, labels in VERB_TO_AT.items():
    VERB_DICT[normalize(k)] = [l.replace("at_", "") for l in labels]

ALL_CLASSES = sorted({l for labels in VERB_DICT.values() for l in labels})
print(f"dictionary: {len(VERB_DICT)} expressions, {len(ALL_CLASSES)} classes")

## Ground truth and evaluation pool

Expression-level GT for Sekaishizuroku. Multi-label is retained in `label_sets`; the primary
(first) label drives stratification and the main single-label evaluation. Classes with fewer than
`MIN_CLASS_SIZE` expressions are excluded from CV and listed separately.

In [ ]:
sekaishi = raw[raw.reference == SRC_SEKAISHI].copy()

gt = {a: VERB_DICT[a] for a in sekaishi.action_norm.unique() if a in VERB_DICT}
not_in_dict = [a for a in sekaishi.action_norm.unique() if a not in VERB_DICT]
print(f"Sekaishi expressions with GT: {len(gt)} / {sekaishi.action_norm.nunique()}")
print(f"Not in dictionary (excluded): {not_in_dict}")

class_counts = Counter(labels[0] for labels in gt.values())
rare_classes = {c for c, n in class_counts.items() if n < MIN_CLASS_SIZE}
print(f"\nRare classes (n < {MIN_CLASS_SIZE}) — excluded from CV, flagged for taxonomy review:")
for c in sorted(rare_classes):
    exprs = [e for e, l in gt.items() if l[0] == c]
    print(f"  {c} (n={class_counts[c]}): {exprs}")

pool = [(e, labels) for e, labels in gt.items() if labels[0] not in rare_classes]
X = np.array([e for e, _ in pool])
y = np.array([labels[0] for _, labels in pool])
label_sets = {e: set(labels) for e, labels in pool}
CV_CLASSES = sorted(set(y))
n_multi = sum(1 for _, labels in pool if len(labels) > 1)
print(f"\nCV pool: {len(X)} expressions, {len(CV_CLASSES)} classes "
      f"({n_multi} multi-label = {n_multi/len(X)*100:.0f}%)")

## Cross Validation Protocol
All methods are evaluated on identical folds (stratified by primary label,
fixed random seed) so that scores and confusion matrices are directly comparable.

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(X, y))

def run_cv(predict_fold):
    """predict_fold(train_idx, test_idx) -> predictions for test_idx (in order)."""
    y_true, y_pred, exprs = [], [], []
    for tr, te in FOLDS:
        y_pred.extend(predict_fold(tr, te))
        y_true.extend(y[te])
        exprs.extend(X[te])
    return np.array(y_true), np.array(y_pred), np.array(exprs)

def summarize(name, y_true, y_pred, exprs):
    """Strict scores use the primary label; relaxed accuracy accepts any label of the expression."""
    y_pred = np.asarray(y_pred)
    return {
        "method": name,
        "macro_F1_%": round(f1_score(y_true, y_pred, average="macro", zero_division=0) * 100, 1),
        "accuracy_%": round(np.mean(y_true == y_pred) * 100, 1),
        "relaxed_acc_%": round(np.mean([p in label_sets[e] for e, p in zip(exprs, y_pred)]) * 100, 1),
        "micro_F1_%": round(f1_score(y_true, y_pred, average="micro", zero_division=0) * 100, 1),
        "UNK_rate_%": round(np.mean(y_pred == "__UNK__") * 100, 1),
    }

## Method 1. Majority class
The majority classifier always predicts the most frequent class of the training fold. It sets the
floor: a method that does not beat it has learned nothing.

In [ ]:
def majority_fold(tr, te):
    maj = Counter(y[tr]).most_common(1)[0][0]
    return [maj] * len(te)

row_maj = summarize("majority", *run_cv(majority_fold))
print(row_maj)

## Method 2. Character n-gram + logistic regression
This is a lexical baseline: TF-IDF over character 1–3-grams feeding a linear classifier.
It sees only surface characters (攻, 撃, 攻撃, …) with no semantics, yet it *generalizes*: an unseen expression like 攻撃開始 shares character features with training
expressions. The gap between this and BERT later quantifies the value of semantic understanding.

In [ ]:
def ngram_fold(tr, te):
    clf = make_pipeline(
        TfidfVectorizer(analyzer="char", ngram_range=(1, 3)),
        LogisticRegression(max_iter=2000),
    )
    clf.fit(X[tr], y[tr])
    return clf.predict(X[te])

row_ng = summarize("char_ngram_logreg", *run_cv(ngram_fold))
print(row_ng)

## Method 3. Japanese WordNet

Each action type is anchored by a list of representative expressions (`CLASS_PROTOTYPES`),
written by hand from the examples of the taxonomy. A test expression is assigned the class
whose anchors have the highest WordNet path similarity to it. WordNet uses no training folds,
so the notebook runs it once over the pool.

**Overlap with the evaluation set.** Most anchors are themselves expressions of the ground
truth (280 of 287, of which 271 under the same primary type), so an expression of the pool can
match itself. With `WN_EXCLUDE_SELF = True`, the classifier ignores an anchor identical to the
test expression. The thesis reports the run with `WN_EXCLUDE_SELF = False`.

In [ ]:
CLASS_PROTOTYPES = {
    "MilitaryAttack": "攻撃 侵攻 侵入 包囲 攻略 撃退 進軍 出兵 上陸 進出",
    "DisputeConflict": "衝突 抗争 参戦 内戦 交戦 武力衝突 戦闘 戦争",
    "ConquestOccupation": "征服 占領 占拠 制圧 支配 領有 駐留",
    "Victory": "勝利 大勝 平定 撃破 完勝 戦勝",
    "Defeat": "敗北 大敗 惨敗 降伏 投降 服属 撤退 敗走",
    "CaptureAndPunishment": "逮捕 捕縛 拉致 配流 処刑 斬首 刑死 捕囚 監禁",
    "AssassinationMassacre": "暗殺 爆殺 射殺 虐殺 粛清 殺害 毒殺 殺戮",
    "Suppression": "弾圧 鎮圧 迫害 排除 禁止 非合法化",
    "RevoltInsurrection": "反乱 蜂起 一揆 暴動 革命 クーデター 挙兵 内戦",
    "ResistanceProtest": "抵抗 抗議 反対 ボイコット デモ 批判 拒否 請願 独立運動",
    "DiplomaticNegotiation": "交渉 調停 仲介 会見 会談 訪問 遣使 謁見 協議",
    "TreatyAgreement": "締結 調印 批准 同盟 和約 和平 承認 受諾 合意",
    "Declaration": "宣言 独立宣言 宣戦布告 通告 布告 発効 声明 表明",
    "DemandOrder": "要求 要請 命令 発令 通達 勧告 最後通牒 警告",
    "ConflictRupture": "対立 断交 破棄 離反 脱退 絶交 決裂 国交断絶",
    "Accession": "即位 戴冠 就任 任命 冊封 選出 当選 再即位",
    "Abdication": "退位 辞職 失脚 退陣 解任 廃位 総辞職 引退",
    "Legislation": "制定 公布 発布 施行 改革 改正 採択 決議 可決",
    "Foundation": "成立 建国 樹立 創設 設立 結成 発足 建立 統一",
    "Beginning": "誕生 開始 登場 勃発 発生",
    "Dissolution": "滅亡 崩壊 解体 解散 廃止 終結 消滅 壊滅 衰亡",
    "Death": "死去 死亡 崩御 戦死 病死 敗死 薨去",
    "IndependenceSeparation": "独立 再独立 分離独立 分裂 自立 分立 脱退 自治",
    "Release": "解放 釈放 返還 撤回 撤退 引き渡し 解除 撤廃 特赦",
    "Ending": "終了 中止 停止 中断 終息 休戦 停戦 収束",
    "Movement": "遷都 移転 移住 移民 出発 到達 漂着 来航 亡命 帰国",
    "Transformation": "改宗 民主化 近代化 国有化 西洋化",
    "Revival": "再建 復活 復興 再興 回復 復帰 再生 再統一",
    "Escalation": "拡大 強化 激化 過激化 増大 躍進 勢力拡大 本格化",
    "ExplorationDiscovery": "探検 発見 航海 世界周航 漂着 調査 登頂 踏査 開拓",
    "ConstructionProduction": "建設 建造 建築 製造 鋳造 着工 完成 作成 開発 竣工",
    "CulturalReligiousExpression": "布教 伝来 普及 刊行 執筆 出版 初演 放送",
    "ScientificInvention": "発明 発見 開発 解明 実験 研究 設計",
    "TradeEconomy": "通商 貿易 販売 輸入 輸出 開港 課税 略奪 採掘",
    "Success": "成功 達成 完遂 実現 完成",
    "Failure": "失敗 挫折 未遂 頓挫",
    "General": "参加 計画 実施 展開 活動 進行 開催 導入 影響 変化",
}
# Restrict anchors to CV classes so predictions stay in the same label space
CV_PROTOTYPES = {c: v for c, v in CLASS_PROTOTYPES.items() if c in CV_CLASSES}


def wn_max_sim(word_a, word_b):
    best = 0.0
    for sa in wn.synsets(word_a, lang="jpn"):
        for sb in wn.synsets(word_b, lang="jpn"):
            s = sa.path_similarity(sb)
            if s and s > best:
                best = s
    return best


WN_EXCLUDE_SELF = False


def wordnet_predict(expr):
    best_cls, best_score = "__UNK__", 0.0
    for cls, vocab in CV_PROTOTYPES.items():
        for proto in vocab.split():
            if WN_EXCLUDE_SELF and normalize(proto) == expr:
                continue
            s = wn_max_sim(expr, proto)
            if s > best_score:
                best_score, best_cls = s, cls
    return best_cls

wn_pred = {}
for i, e in enumerate(X):
    wn_pred[e] = wordnet_predict(e)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(X)}")

yt_wn = y.copy()
yp_wn = np.array([wn_pred[e] for e in X])
row_wn = summarize("japanese_wordnet", yt_wn, yp_wn, X)
print(row_wn)

## Method 4. Tohoku BERT v3

**Tohoku BERT v3** (`cl-tohoku/bert-base-japanese-v3`), the Japanese BERT used during the
construction of the taxonomy. The notebook uses the [CLS] vector, as in the construction.

1. **Classification**: kNN (k = 5) and nearest centroid over the cross-validation folds.
2. **Clustering**: K-means with as many clusters as types, on the evaluation pool alone and on
   the combined expressions of both sources (as in the construction), scored with AMI and ARI
   on the expressions of the pool.

### Encoder

In [ ]:
def encode_tohoku(texts):
    """Encode texts as Tohoku BERT v3 [CLS] vectors (768-dim). Matches vectorize.ipynb."""
    import torch
    from transformers import BertJapaneseTokenizer, BertModel
    print("  loading Tohoku BERT v3 ...")
    tokenizer = BertJapaneseTokenizer.from_pretrained("cl-tohoku/bert-base-japanese-v3")
    model = BertModel.from_pretrained("cl-tohoku/bert-base-japanese-v3")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    print(f"  device: {device}")
    vecs = []
    with torch.no_grad():
        for i, t in enumerate(texts):
            inputs = tokenizer(t, return_tensors="pt").to(device)
            out = model(**inputs)
            vecs.append(out.last_hidden_state[0][0].cpu().numpy())  # [CLS]
            if (i + 1) % 100 == 0:
                print(f"  {i+1}/{len(texts)}")
    return np.vstack(vecs).astype(np.float32)


def get_embeddings(tag, texts):
    """Load from cache or encode. tag: 'pool' or 'combined'."""
    cache = Path(f"emb_tohoku_{tag}.npy")
    if cache.exists():
        v = np.load(cache)
        print(f"[tohoku/{tag}] loaded cache {v.shape}")
    else:
        print(f"[tohoku/{tag}] encoding {len(texts)} expressions ...")
        v = encode_tohoku(list(texts))
        np.save(cache, v)
        print(f"[tohoku/{tag}] saved {v.shape}")
    return v / np.linalg.norm(v, axis=1, keepdims=True)  # L2-normalize

In [ ]:
# Encode the evaluation pool (478 expressions)
emb_pool = get_embeddings("pool", X)
print("pool embeddings ready:", emb_pool.shape)

### kNN and nearest centroid

In [ ]:
def knn_fold(tr, te):
    sims = emb_pool[te] @ emb_pool[tr].T
    preds = []
    for row in sims:
        top = np.argsort(row)[::-1][:K_NEIGHBORS]
        preds.append(Counter(y[tr][top]).most_common(1)[0][0])
    return preds

def centroid_fold(tr, te):
    cents, cls = [], []
    for c in CV_CLASSES:
        idx = tr[y[tr] == c]
        if len(idx) == 0:
            continue
        v = emb_pool[idx].mean(axis=0)
        cents.append(v / np.linalg.norm(v)); cls.append(c)
    sims = emb_pool[te] @ np.vstack(cents).T
    return [cls[i] for i in sims.argmax(axis=1)]

row_knn = summarize("tohoku_knn_k5", *run_cv(knn_fold))
row_cen = summarize("tohoku_centroid", *run_cv(centroid_fold))
print(row_knn)
print(row_cen)

### Clustering
K-means with k = number of types in the pool. AMI is the main measure and ARI is reported
alongside. Both are computed on the expressions of the pool against their primary label.

In [ ]:
K = len(CV_CLASSES)
pool_index = {e: i for i, e in enumerate(X)}

def cluster_scores(name, emb, exprs):
    labels = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10).fit_predict(emb)
    idx = [i for i, e in enumerate(exprs) if e in pool_index]
    pred = labels[idx]
    gold = [y[pool_index[exprs[i]]] for i in idx]
    return {"set": name, "clustered": len(exprs), "scored": len(idx),
            "AMI": round(adjusted_mutual_info_score(gold, pred), 3),
            "ARI": round(adjusted_rand_score(gold, pred), 3)}

# Combined expressions of both sources, as in the construction of the taxonomy
combined = np.array(sorted(raw[raw.reference.isin([SRC_SEKAISHI, SRC_FUNSO])].action_norm.unique()))
emb_combined = get_embeddings("combined", combined)

table9 = pd.DataFrame([cluster_scores("pool", emb_pool, X),
                       cluster_scores("both sources", emb_combined, combined)])
table9.to_csv(OUT_DIR / "clustering_scores.csv", index=False)
display(table9)

## Method 5. LLM classification

The LLMs received the names of the types and representative expressions (`prompt.md`), without
demonstrations or training data. The answers were collected through chat interfaces in batches and
saved as one text file per model. This section parses those files, scores each model on the same
pool, and measures the agreement between the models.

**Before running:** place the response files in `DATA_DIR` and register them in `MODEL_FILES`.

Outputs: `prompt.md`, and in `results/`: `benchmark_scores.csv`, `llm_agreement.csv`, `llm_unanimous_wrong.csv`,
`prompt_overlap_scores.csv`, `confusion_llm_best.png`.

### Prompt given to the LLMs
The setup prompt and the catalog of types, as pasted into each chat before the batches of
expressions. The catalog lists the types of the evaluation pool with the same representative
expressions as the WordNet anchors (`CV_PROTOTYPES`). The cell writes the prompt to `prompt.md`.

In [ ]:
CATALOG = "\n".join(f"- {c}: {CV_PROTOTYPES[c]}" for c in CV_CLASSES)

SETUP_PROMPT = f"""You classify Japanese historical action expressions into exactly one ActionType class.
The classes and their typical definition verbs are:

{CATALOG}

Rules:
- I will send you batches of expressions as JSON arrays.
- For each batch, answer with a single JSON object mapping each input expression to one
  class name.
- Use only class names from the list above, verbatim.
- Choose the single most appropriate class for each expression.
- Output ONLY the JSON object, no explanations, no code fences.

Reply "ready" if you understand."""

PROMPT_FILE.write_text(SETUP_PROMPT + "\n", encoding="utf-8")
print(f"wrote {PROMPT_FILE} ({len(CV_CLASSES)} types)")

In [ ]:
MODEL_FILES = {
    "llm_model_sonnet": DATA_DIR / "llm_responses_sonnet.txt",
    "llm_model_chatgpt": DATA_DIR / "llm_responses_chatgpt.txt",
    "llm_model_fable": DATA_DIR / "llm_responses_fable.txt",
    "llm_model_geminiflash": DATA_DIR / "llm_responses_geminiflash.txt",
}
for name, f in MODEL_FILES.items():
    print(f"  {name}: {f} exists={f.exists()}")

### Parse response files
Each file contains 10 JSON objects (one per batch), possibly with chat text between
them. The parser extracts every top-level JSON object and merges them. Predictions are
normalized: unknown class names → `__UNK__`; missing expressions → `__UNK__`. A coverage report
per model shows how many of the 478 expressions were answered.

In [ ]:
def parse_responses(path):
    """Extract and merge all top-level JSON objects from a chat response dump."""
    text = Path(path).read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    preds, i = {}, 0
    while i < len(text):
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            if isinstance(obj, dict):
                preds.update(obj)
            i = start + end
        except json.JSONDecodeError:
            i = start + 1
    return preds


MODEL_PREDS = {}
for name, f in MODEL_FILES.items():
    if not f.exists():
        print(f"[{name}] SKIPPED, file not found: {f}")
        continue
    raw_preds = parse_responses(f)
    # normalise expressions (keys) the same way as the pool, validate classes
    preds = {normalize(k): (v if v in CV_CLASSES else "__UNK__") for k, v in raw_preds.items()}
    covered = sum(1 for e in X if e in preds)
    print(f"[{name}] parsed {len(raw_preds)} predictions, pool coverage {covered}/{len(X)}")
    missing = [e for e in X if e not in preds]
    if missing:
        print(f"  missing ({len(missing)}): {missing[:10]}{' ...' if len(missing) > 10 else ''}")
    MODEL_PREDS[name] = preds

### Scores per model and majority vote

In [ ]:
llm_rows, pred_arrays = [], {}
for name, preds in MODEL_PREDS.items():
    yp = np.array([preds.get(e, "__UNK__") for e in X])
    pred_arrays[name] = yp
    llm_rows.append(summarize(name, y, yp, X))

# Majority vote over the models (ties go to the answer of the first model listed among the tied)
if len(pred_arrays) >= 2:
    names = list(pred_arrays)
    yp_ens = []
    for i in range(len(X)):
        votes = Counter(pred_arrays[n][i] for n in names if pred_arrays[n][i] != "__UNK__")
        if votes:
            top_count = votes.most_common(1)[0][1]
            tied = [c for c, n_ in votes.items() if n_ == top_count]
            winner = next((pred_arrays[n][i] for n in names if pred_arrays[n][i] in tied), tied[0])
        else:
            winner = "__UNK__"
        yp_ens.append(winner)
    pred_arrays["ensemble_majority"] = np.array(yp_ens)
    llm_rows.append(summarize("ensemble_majority", y, pred_arrays["ensemble_majority"], X))

## Comparison of all classifiers (Table 8)

In [ ]:
scores = pd.DataFrame([row_maj, row_ng, row_wn, row_knn, row_cen] + llm_rows)
scores.to_csv(OUT_DIR / "benchmark_scores.csv", index=False)
display(scores[["method", "macro_F1_%", "accuracy_%", "relaxed_acc_%", "UNK_rate_%"]])

### Agreement between the models
Pairwise raw agreement (% of expressions on which two models give the same type) and
**Fleiss' κ** over all models (chance-corrected agreement for several raters, single label).

The thesis reads κ after Carletta (1996): above 0.8 good reliability, between 0.67 and 0.8
tentative conclusions.

Two points to keep in mind:
1. The LLMs are not independent annotators, since they share training data and biases.
2. High agreement on a wrong type is still agreement, so κ needs reading together with
   accuracy. Expressions on which all models agree with one another and disagree with the
   ground truth point to places where the ground truth or the taxonomy deserves review.

In [ ]:
model_names = [n for n in pred_arrays if n != "ensemble_majority"]

# Pairwise agreement matrix
agree = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
for a, b in combinations(model_names, 2):
    agree.loc[a, b] = agree.loc[b, a] = round(np.mean(pred_arrays[a] == pred_arrays[b]) * 100, 1)
for n in model_names:
    agree.loc[n, n] = 100.0
agree.to_csv(OUT_DIR / "llm_agreement.csv")
print("pairwise agreement (%):")
display(agree)


def fleiss_kappa(item_idx):
    """Fleiss' kappa over model_names for the expressions at item_idx."""
    cats = CV_CLASSES + ["__UNK__"]
    cat_idx = {c: i for i, c in enumerate(cats)}
    n_raters = len(model_names)
    counts = np.zeros((len(item_idx), len(cats)))
    for name in model_names:
        for row, i in enumerate(item_idx):
            counts[row, cat_idx[pred_arrays[name][i]]] += 1
    P_i = ((counts ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    p_j = counts.sum(axis=0) / (len(item_idx) * n_raters)
    P_e = (p_j ** 2).sum()
    return (P_i.mean() - P_e) / (1 - P_e)

ALL_IDX = np.arange(len(X))
print(f"Fleiss' kappa over {len(model_names)} models: {fleiss_kappa(ALL_IDX):.3f}")

# Unanimous errors (strict): all models give the same type and it differs from the primary label
unanimous_wrong = []
for i, e in enumerate(X):
    answers = {pred_arrays[n][i] for n in model_names}
    if len(answers) == 1:
        p = answers.pop()
        if p != "__UNK__" and p != y[i]:
            kind = "second type of the ground truth" if p in label_sets[e] else "not in the ground truth"
            unanimous_wrong.append((e, sorted(label_sets[e]), p, kind))
uw = pd.DataFrame(unanimous_wrong, columns=["expression", "gold", "all_models_say", "kind"])
uw.to_csv(OUT_DIR / "llm_unanimous_wrong.csv", index=False)
print(f"unanimous errors (strict): {len(uw)}")
print(uw.kind.value_counts().to_string())

## Expressions in and out of the prompt
The prompt lists representative expressions for each type, and some of them belong to the
evaluation pool. The scores are split by whether an expression appears in that list.

In [ ]:
def prompt_expressions(path):
    """Expressions listed in the prompt, split on the usual separators and normalised."""
    text = normalize(Path(path).read_text(encoding="utf-8"))
    tokens = re.split(r"[・、,，\s/|:：;；()（）\[\]「」『』\"']+", text)
    return {t for t in tokens if t}

PROMPT_EXPRS = prompt_expressions(PROMPT_FILE)
in_prompt = np.array([e in PROMPT_EXPRS for e in X])
print(f"in prompt: {in_prompt.sum()}, not in prompt: {(~in_prompt).sum()}")

rows = []
for subset, mask in [("all", np.ones(len(X), bool)), ("in prompt", in_prompt), ("not in prompt", ~in_prompt)]:
    idx = np.where(mask)[0]
    row = {"subset": subset, "n": len(idx), "Fleiss_kappa": round(fleiss_kappa(idx), 3)}
    for name in model_names:
        row[f"{name}_macro_F1_%"] = round(
            f1_score(y[idx], pred_arrays[name][idx], average="macro", zero_division=0) * 100, 1)
    rows.append(row)
table10 = pd.DataFrame(rows)
table10.to_csv(OUT_DIR / "prompt_overlap_scores.csv", index=False)
display(table10)

if len(uw):
    uw_out = uw[uw.kind == "not in the ground truth"].expression.map(lambda e: e not in PROMPT_EXPRS)
    print(f"unanimous errors outside the ground truth: {len(uw_out)}, of which not in prompt: {uw_out.sum()}")

### Confusion matrix of the best model
The confusion matrix of the LLM with the highest Macro-F1 shows which types the models confuse,
the material for the qualitative error analysis of section 4.2.

In [ ]:
llm_scores = pd.DataFrame(llm_rows)
best_name = (llm_scores[~llm_scores.method.str.startswith("ensemble")]
             .sort_values("macro_F1_%", ascending=False).iloc[0]["method"])
yp_best = pred_arrays[best_name]
print("best single LLM:", best_name)

def plot_confusion(y_true, y_pred, title, fname):
    classes = CV_CLASSES
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(13, 12))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=90, fontsize=7)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=7)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j] > 0:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, fraction=0.046)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

plot_confusion(y, yp_best, f"{best_name}: confusion matrix", OUT_DIR / "confusion_llm_best.png")